# market basket analysis - groceries

AI1030 assignment. needs `groceries.csv` in the same folder (one basket per line, items separated by commas).

In [ ]:
import itertools
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

SEED = 42        # for the random prices
MIN_COUNT = 20   # min number of baskets for a pair/triple to count
K = 10           # how many top pairs/triples to show

## A. load + basic stats

In [ ]:
transactions_raw = []
with open("groceries.csv") as f:
    for line in f:
        items = [x.strip() for x in line.strip().split(",") if x.strip()]
        transactions_raw.append(items)

unique_products = set(itertools.chain.from_iterable(transactions_raw))
basket_sizes = [len(t) for t in transactions_raw]

print("transactions:", len(transactions_raw))
print("unique products:", len(unique_products))
print("median basket size:", int(np.median(basket_sizes)))
print("95th percentile:", int(np.percentile(basket_sizes, 95)))

counts = Counter(itertools.chain.from_iterable(transactions_raw))
print("\ntop 20:")
for item, c in counts.most_common(20):
    print(f"  {item}: {c}")

## B. cleaning

lowercase, spaces -> underscores, remove duplicate items inside the same basket, drop baskets with less than 2 items (they can't form pairs anyway)

In [ ]:
transactions_clean = []
for items in transactions_raw:
    std = [x.lower().strip().replace(" ", "_") for x in items]
    std = list(dict.fromkeys(x for x in std if x))  # dedupe, keep order
    if len(std) >= 2:
        transactions_clean.append(std)

transactions = pd.DataFrame({
    "transaction_id": range(1, len(transactions_clean) + 1),
    "items": transactions_clean,
})
transactions["basket_size"] = transactions["items"].apply(len)

print("baskets after cleaning:", len(transactions))
print("unique products after cleaning:", len(set(itertools.chain.from_iterable(transactions["items"]))))
transactions.head()

## C. random prices + basket totals

In [ ]:
products = sorted(set(itertools.chain.from_iterable(transactions["items"])))
rng = np.random.default_rng(SEED)
price_df = pd.DataFrame({"product": products, "price": np.round(rng.uniform(0.5, 15.0, len(products)), 2)})
price = dict(zip(price_df["product"], price_df["price"]))

transactions["basket_total"] = transactions["items"].apply(lambda items: round(sum(price[x] for x in items), 2))
transactions[["transaction_id", "items", "basket_total"]].head()

## D. pairs and triples

sort each basket first so (a, b) and (b, a) count as the same pair

In [ ]:
pair_counter = Counter()
triple_counter = Counter()
for items in transactions["items"]:
    pair_counter.update(itertools.combinations(sorted(items), 2))
    triple_counter.update(itertools.combinations(sorted(items), 3))

n = len(transactions)

def top_itemsets(counter):
    rows = [{"itemset": s, "support_count": c, "support_fraction": c / n}
            for s, c in counter.items() if c >= MIN_COUNT]
    if not rows:
        return pd.DataFrame(columns=["itemset", "support_count", "support_fraction"])
    df = pd.DataFrame(rows)
    return df.sort_values(["support_fraction", "itemset"], ascending=[False, True]).head(K)

pairs_df = top_itemsets(pair_counter)
triples_df = top_itemsets(triple_counter)

print("top pairs:")
print(pairs_df.to_string(index=False))
print("\ntop triples:")
print(triples_df.to_string(index=False) if len(triples_df) else "  none above MIN_COUNT")

## E. plots

In [ ]:
import os
os.makedirs("figures", exist_ok=True)

def save_and_show(name):
    plt.tight_layout()
    plt.savefig(f"figures/{name}.png", dpi=120)
    plt.show()

item_counts = Counter(itertools.chain.from_iterable(transactions["items"]))

# top 15 items
names, vals = zip(*item_counts.most_common(15))
plt.figure()
plt.bar(range(15), vals)
plt.xticks(range(15), names, rotation=90)
plt.ylabel("frequency")
plt.title("top 15 items")
save_and_show("figure_top15_items")

# top pairs
if len(pairs_df):
    labels = [" & ".join(p) for p in pairs_df["itemset"]]
    plt.figure()
    plt.bar(range(len(labels)), pairs_df["support_fraction"])
    plt.xticks(range(len(labels)), labels, rotation=90)
    plt.ylabel("support fraction")
    plt.title(f"top {K} pairs")
    save_and_show("figure_top_pairs")

# co-occurrence heatmap for the 25 most common items
top25 = [x for x, _ in item_counts.most_common(25)]
sets = [set(items) for items in transactions["items"]]
matrix = np.zeros((25, 25))
for i, a in enumerate(top25):
    for j, b in enumerate(top25):
        matrix[i, j] = sum(1 for s in sets if a in s and b in s) / n

plt.figure()
plt.imshow(matrix, aspect="auto")
plt.colorbar(label="support fraction")
plt.xticks(range(25), top25, rotation=90)
plt.yticks(range(25), top25)
plt.title("co-occurrence (top 25 items)")
save_and_show("figure_cooccurrence_heatmap")

plt.figure()
plt.hist(transactions["basket_size"], bins=20)
plt.xlabel("basket size")
plt.title("basket size distribution")
save_and_show("figure_basket_size_hist")

plt.figure()
plt.hist(transactions["basket_total"], bins=20)
plt.xlabel("basket total ($)")
plt.title("basket total distribution")
save_and_show("figure_basket_total_hist")